<a href="https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Task type: Ranking (with a scoring/classification component underneath).**

The actual decision — "which pages should the content team refresh first" —
isn't "is this page declining, yes/no" in isolation. It's "given a queue of
thousands of pages, put the ones worth fixing first at the top." That's a
ranking problem: the output people act on is an ordered list, not a single
label. Underneath it, I'll likely train a classifier or scorer that predicts
a declining-probability per page, then rank pages by that score — the same
pattern the Week 1 pipeline used (random forest score -> sorted queue,
evaluated with Precision@50).

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` — whether a page's traffic trend is
classified as "down" (derived from `trend_direction` / `trend_pct` in the
starter data).

**Where it comes from:** it's a **defined rule**, not a directly observed
outcome. The dataset buckets a continuous percentage change (`trend_pct`)
into categories like up/down/flat/stable/new, and the label is `1` when that
bucket is "down." That matters: it's a proxy for "this page needs attention,"
not a ground-truth measurement of business harm. A page could be labeled
"down" by a small threshold crossing while still performing fine in absolute
terms, or vice versa — so the label is a reasonable stand-in for the real
target, not the real target itself.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["trend_direction"].value_counts())
print("\nis_declining_label distribution:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label distribution:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50** (same one used in Weeks 1–2).

Of the top 50 pages my ranking puts at the front of the queue, what fraction
are actually declining? I'm defending this over plain accuracy because the
content team only ever acts on the top of the list — a model that's 95%
accurate overall but wrong about who belongs in the top 50 is useless to
them. Precision@K matches how the output is actually consumed: a short,
ordered action list, not a verdict on every page in the dataset.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# sanity check using a naive "proxy score" just to confirm the function works
naive_score = df["impressions_90d"]
y = df["is_declining_label"].values
print(f"Naive impressions-only ranking, Precision@50: {precision_at_k(naive_score, y, 50):.3f}")

Naive impressions-only ranking, Precision@50: 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one page**, at a point-in-time snapshot (its current content
age, position, impressions, CTR, word count, and the trend label computed
from recent history). The queue I'd rank is exactly this table, sorted by a
predicted score instead of any single raw column.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_age_days", "days_since_last_update", "impressions_90d",
        "avg_position", "ctr", "word_count", "position_tier",
        "trend_direction", "is_declining_label"]

lane_slice = df[cols].copy()
print("Rows (pages):", len(lane_slice))
lane_slice.head(5)

Rows (pages): 30000


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,position_tier,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,striking,down,1
1,445,25,15320,20.3,0.05,2481.0,page_3_5,down,1
2,141,20,12581,36.5,0.09,3515.0,page_3_5,down,1
3,463,22,11751,6.2,0.49,NaN,page_1,stable,0
4,263,14,19140,44.0,0.13,2803.0,page_3_5,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Week 2 already showed this empirically: a hand rule (`stale AND visible`,
ranked by impressions) hit Precision@20 = 0.900 but only Precision@50 = 0.680.
A depth-2 tree — just three yes/no questions — split on different features
than my intuition would have picked (`impressions_90d` first, then
`avg_position` or `content_age_days`), and the two approaches traded wins
depending on where in the list you looked.

The pattern is too messy for a single if-statement because "worth
refreshing" depends on an *interaction* of signals, not one threshold: a
stale page with low visibility isn't worth fixing, and a fresh page with
falling CTR still might be. A fixed rule has to pick one or two hard
cutoffs by hand; a model can weigh several signals jointly and find where
their combination actually separates declining pages from the rest — which
is exactly why the tree beat the hand rule further down the ranked list,
where simple rules run out of signal.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier, export_text

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

for k in (20, 50):
    print(f"Precision@{k}:  hand rule {precision_at_k(hand_rule_score, y, k):.3f}"
          f"   vs   tree {precision_at_k(tree_score, y, k):.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.